<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**The queue: what to do first, and why.**

This playbook turns the validated four-feature Random Forest (precision@50 = 0.760, base rate 47.7%, client-grouped split) into a ranked review queue. Every page carries one reason code and one recommended action. A reviewer works the list top to bottom and stops when capacity runs out.

**Reason codes and actions, in priority order:**

| Priority | Reason code | Action | Plain-language why |
|---|---|---|---|
| 1 | `weak_position_high_demand` | **Refresh** — rewrite, expand, update | Page has real search demand (high impressions) but ranks outside top 10. The content isn't capturing the opportunity its topic creates. |
| 2 | `low_ctr_good_position` | **Optimize title & meta** — tighten the snippet | Page ranks well but converts impressions to clicks poorly. The ranking is fine; the promise in the search result isn't compelling enough. |
| 3 | `thin_engagement` | **Review manually** — diagnose before acting | Page has impressions but very few clicks relative to volume. Could be a targeting mismatch, a cannibalised query, or a page that serves a purpose clicks don't measure (e.g. branded navigation). Human judgment needed before committing effort. |

**What the score means.** The score is a probability estimate from the Random Forest — higher means the model considers this page more likely to be declining. It is *not* a severity measure, a priority weight, or a confidence score. Two pages with scores 0.82 and 0.81 are not meaningfully different; the ranking matters, the exact number doesn't.

**What "declining" means in this context.** The label is: did the daily click rate fall between the first 21 days and the last 10 days of the month. It is a proxy for short-term momentum loss, not a diagnosis of content quality. A page can be "declining" by this measure because of seasonality, a one-off traffic spike ending, or normal fluctuation — not just because the content got worse.

In [ ]:
!pip install -q duckdb

import duckdb, pandas as pd, numpy as np, json, os
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
HF_TOKEN = userdata.get("HF_TOKEN").strip()
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL       = "hf://datasets/FlyRank/internship-warehouse"
CONTENT   = f"{REL}/dim_content.parquet"
DEV_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
FEAT_END, OUT_START = "2026-03-21", "2026-03-22"

frame = con.sql(f"""
    WITH feat AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)            AS impressions_21d,
               SUM(gsc_clicks)                 AS clicks_21d,
               AVG(NULLIF(gsc_avg_position,0)) AS avg_position_21d,
               COUNT(*)                        AS days_observed
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date <= DATE '{FEAT_END}'
        GROUP BY 1,2
        HAVING SUM(gsc_clicks) > 0
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_out, COUNT(*) AS days_out
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date >= DATE '{OUT_START}'
        GROUP BY 1,2
    )
    SELECT f.*, o.clicks_out, o.days_out
    FROM feat f
    JOIN outcome o USING (client_hash_id, content_hash_id)
    WHERE o.days_out > 0 AND o.clicks_out > 0
""").df()

frame["ctr_21d"]      = frame.clicks_21d / frame.impressions_21d
frame["rate_before"]  = frame.clicks_21d / frame.days_observed
frame["rate_after"]   = frame.clicks_out / frame.days_out
frame["is_declining"] = (frame.rate_after < frame.rate_before).astype(int)

# Four-feature model (clicks_21d dropped per ML-09 audit)
FEATURES = ["impressions_21d", "avg_position_21d", "ctr_21d", "days_observed"]
X, y, groups = frame[FEATURES], frame.is_declining, frame.client_hash_id

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(X, y, groups))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
rf.fit(X.iloc[tr].fillna(-1), y.iloc[tr])

# Score the full frame for the queue
frame["score"] = rf.predict_proba(X.fillna(-1))[:, 1]

# Reason codes (same logic as ML-07, applied to the scored frame)
def assign_reason(row):
    if row.avg_position_21d > 10 and row.impressions_21d >= frame.impressions_21d.quantile(0.10):
        return "weak_position_high_demand"
    if row.avg_position_21d <= 10 and row.ctr_21d < 0.0038:
        return "low_ctr_good_position"
    if row.impressions_21d >= 100 and row.ctr_21d < 0.01:
        return "thin_engagement"
    return "no_flag"

frame["reason_code"] = frame.apply(assign_reason, axis=1)

ACTION = {
    "weak_position_high_demand": "refresh",
    "low_ctr_good_position":     "optimize_title_meta",
    "thin_engagement":           "review_manually",
    "no_flag":                   "no_action",
}
frame["action"] = frame.reason_code.map(ACTION)

queue = frame.sort_values(["score", "content_hash_id"], ascending=[False, True]).reset_index(drop=True)

print(f"Queue: {len(queue):,} pages")
print(queue.reason_code.value_counts())

def precision_at_k(k):
    return queue.head(k).is_declining.mean()

print(f"\nBase rate: {y.mean():.1%}")
for k in (20, 50):
    print(f"precision@{k}: {precision_at_k(k):.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue: 36,290 pages
reason_code
low_ctr_good_position        13189
thin_engagement              10227
weak_position_high_demand     8847
no_flag                       4027
Name: count, dtype: int64

Base rate: 44.4%
precision@20: 100.0%
precision@50: 92.0%


In [ ]:
# Honest metric — test set only
test_queue = frame.iloc[te].sort_values(["score", "content_hash_id"], ascending=[False, True])
print(f"\nTest-set only ({len(test_queue):,} pages):")
for k in (20, 50):
    print(f"  precision@{k}: {test_queue.head(k).is_declining.mean():.1%}")


Test-set only (18,174 pages):
  precision@20: 70.0%
  precision@50: 76.0%


## 2. Intended use and limits

**Who uses this:** a content reviewer or SEO editor working a capacity-limited queue — the same person described in ML-02. They open the top-ranked pages, read the reason code, and decide whether to act.

**What it's for:** decision-support. The queue orders attention; it does not take action. Every page still requires a human judgment call before any edit, redirect, or consolidation happens.

**Where it stops being valid:**

- **Population:** the queue covers 36,290 pages — the 40.7% of the GSC-available March frame where both feature-window and outcome-window clicks are non-zero. The remaining 59.3% are invisible to this model. Pages with zero clicks in either window are not scored, not ranked, and not safe to assume are fine.
- **Time:** trained and scored on March 2026 only. Seasonal patterns, algorithm updates, and client-specific campaigns are not represented. A queue built in March may not order correctly in December.
- **Clients:** validated on 12 held-out clients (precision@50 = 0.720, base rate 47.7%). It has never seen clients outside the 40 in this frame. A new client with a different content profile may behave differently.
- **Label:** the proxy is "did the daily click rate fall between days 1–21 and days 22–31." That captures short-term momentum loss, not content quality. A page can decline by this measure because of seasonality, a spike ending, or normal noise — not just because the content got worse.
- **Not causal:** the model observes association between features and decline. It does not say refreshing a flagged page will recover it. That claim needs an experiment this data cannot support.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

**What a person must check before acting on a flagged page:**

- **Read the page, not just the score.** The reason code says *why* the model flagged it (bad position, low CTR, thin engagement). The reviewer decides whether that diagnosis matches what the page actually looks like. A page flagged `low_ctr_good_position` that turns out to be a navigation page with no click intent is a false alarm — the model can't know intent from these features.
- **Check the traffic source.** A page whose clicks come mostly from branded queries behaves differently from one earning non-branded organic clicks. The model doesn't distinguish these.
- **Check recency.** The model is trained on March 2026 data. If the page has been updated since then, the flag may be stale.
- **Check whether the page is part of a group.** Multiple pages on the same topic may cannibalise each other. Refreshing one without consolidating the others can make the problem worse.

**The no-go list — what should never be automated:**

- **Publishing or unpublishing a page.** The queue ranks; a human decides whether to act and how.
- **Redirecting or merging pages.** Consolidation decisions require editorial judgment about which page should survive.
- **Changing page titles or meta descriptions at scale.** `optimize_title_meta` is a suggestion to a reviewer, not a batch instruction to a script.
- **Treating `no_flag` pages as safe.** In ML-07 we measured that `no_flag` pages had the *highest* decline rate (80.0% on the starter slice). The model assigns them low scores because they don't match the reason-code conditions, not because they're healthy. Absence of a flag is not evidence of health.
- **Running the queue on a different month without retraining.** The model's feature distributions are calibrated to March 2026. Scoring April data with a March model is extrapolation, not prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

**What would tell you the recommendations went stale?**

- **Precision@50 on the next month's holdout drops below base rate.** That's the clearest signal the model has stopped being useful. Check monthly by scoring the new month's pages, waiting for the outcome window, and computing precision@K on a fresh client-grouped split.
- **Base rate shifts by more than 10 percentage points.** The current base rate is 44.4%. If a future month's rate is 30% or 55%, the population has changed and the model's calibration is off.
- **A new client type enters the portfolio.** The model was validated on 12 held-out clients from 40 total. A client in a domain the training set never covered (e.g. e-commerce if the current portfolio is content-heavy) may behave differently.
- **Feature distributions shift.** If `avg_position_21d` or `impressions_21d` suddenly look very different from March — because of a Google algorithm update, a seasonal spike, or a new content push — the model's splits are calibrated to the wrong ranges.
- **The reason codes stop matching reviewer experience.** If reviewers consistently disagree with the top-10 flags — "these pages aren't really worth reviewing" — the queue is no longer ordering usefully, regardless of what precision@K says.

**Retrain, don't patch.** When any trigger fires, retrain on the new month's data rather than adjusting thresholds. The model is cheap to retrain (four features, one Random Forest, one month of data), and patching a stale model is how invisible drift accumulates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

**Exports for the paper.** Three files written to `work/outputs/` and `work/figures/`:

- `playbook_queue.csv` — the full ranked queue (36,290 pages) with reason codes, actions, and scores. This is what the paper's recommendations section builds on.
- `playbook_metrics.json` — the model's receipts: features used, base rate, test-set precision@K, reason-code counts. Every number in the paper traces back to this file.
- `reason_code_dist.png` — a figure showing the distribution of reason codes across the queue, reusable in the paper.

The CSV stays out of git (CI leak-guard blocks data files). The JSON and figure get committed.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export the ranked queue
cols = ["client_hash_id", "content_hash_id", "reason_code", "action", "score",
        "impressions_21d", "avg_position_21d", "ctr_21d", "days_observed", "is_declining"]
queue[cols].to_csv("work/outputs/playbook_queue.csv", index=False)
print(f"Wrote queue: {len(queue):,} rows → work/outputs/playbook_queue.csv")

# 2. Export metrics JSON (the paper's receipts)
metrics = {
    "model": "RandomForest_4feat",
    "features": FEATURES,
    "month": "2026-03",
    "frame_pages": len(frame),
    "base_rate": round(y.mean(), 3),
    "test_clients": int(frame.iloc[te].client_hash_id.nunique()),
    "test_pages": len(te),
    "test_base_rate": round(y.iloc[te].mean(), 3),
    "precision_at_20_test": round(test_queue.head(20).is_declining.mean(), 3),
    "precision_at_50_test": round(test_queue.head(50).is_declining.mean(), 3),
    "reason_code_counts": queue.reason_code.value_counts().to_dict(),
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote metrics → work/outputs/playbook_metrics.json")
print(json.dumps(metrics, indent=2))

# 3. Export a reason-code distribution figure
fig, ax = plt.subplots(figsize=(7, 4))
queue.reason_code.value_counts().plot.barh(ax=ax, color="#4a7c91")
ax.set_xlabel("Pages")
ax.set_title("Reason code distribution (playbook queue)")
fig.tight_layout()
fig.savefig("work/figures/reason_code_dist.png", dpi=150)
print("Wrote figure → work/figures/reason_code_dist.png")
plt.close()

Wrote queue: 36,290 rows → work/outputs/playbook_queue.csv
Wrote metrics → work/outputs/playbook_metrics.json
{
  "model": "RandomForest_4feat",
  "features": [
    "impressions_21d",
    "avg_position_21d",
    "ctr_21d",
    "days_observed"
  ],
  "month": "2026-03",
  "frame_pages": 36290,
  "base_rate": 0.444,
  "test_clients": 11,
  "test_pages": 18174,
  "test_base_rate": 0.477,
  "precision_at_20_test": 0.7,
  "precision_at_50_test": 0.76,
  "reason_code_counts": {
    "low_ctr_good_position": 13189,
    "thin_engagement": 10227,
    "weak_position_high_demand": 8847,
    "no_flag": 4027
  }
}
Wrote figure → work/figures/reason_code_dist.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.